# E01_GridWorld — табличный Q-learning

**Урок:** 1.3–1.4 курса [cyber-unity-learn](https://rl-cuber-unity-code.com/)  
**Что реализуется:** табличный Q-learning в среде Unity + Value Iteration как точный эталон.  
**Ожидаемый результат:** средняя награда **0.68** — доказанный оптимум среды.  
**Время обучения:** ~70 секунд на сид (`QUICK_RUN = True` — около 8 секунд).

## Почему этот пример стоит первым

Среда достаточно мала, чтобы **оптимум был известен точно**: Value Iteration на
модели MDP даёт `Q*` за 10 итераций. Поэтому здесь можно не гадать, «хорошо ли
обучился агент», а сравнить его политику с доказанно оптимальной. Такой
ориентир исчезает во всех последующих примерах — им стоит воспользоваться.

Второй смысл примера: он проверяет **связку Python ↔ Unity целиком**, включая
экспорт в ONNX, но без нейросети — если что-то сломается, дело точно не в сети.

## Требования

Собранный билд среды:

```powershell
python scripts\build_env.py E01_GridWorld
```

In [1]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)

print(f"python        {platform.python_version()}")
print(f"numpy         {np.__version__}")
print(f"torch         {torch.__version__}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {resolve_device('auto')}  (табличному методу GPU не нужен)")
print(f"сид           {SEED}")

python        3.10.11
numpy         1.23.5
torch         2.2.1+cu121
mlagents_envs 1.2.0.dev0
устройство    cuda  (табличному методу GPU не нужен)
сид           0


## 2. Конфигурация

Единственный блок со всеми гиперпараметрами. Правьте здесь — ниже ничего
менять не нужно.

In [2]:
QUICK_RUN = True   # True — быстрая проверка (~8 с); False — полное обучение

from labrl.algos.tabular.q_learning import QLearningConfig
from labrl.train.tabular import TabularTrainConfig
from labrl.utils.schedules import LinearSchedule

ALGO_CFG = QLearningConfig(
    gamma=0.99,            # горизонт дисконтирования
    learning_rate=0.2,     # альфа табличного обновления
    initial_q=0.0,         # нейтральная инициализация
)

TOTAL_STEPS = 2_000 if QUICK_RUN else 20_000

TRAIN_CFG = TabularTrainConfig(
    total_steps=TOTAL_STEPS,
    eval_every_steps=TOTAL_STEPS // 5,
    eval_episodes=20,
)

# Разведка: линейный спад ε за 60% бюджета.
EPSILON = LinearSchedule(start=1.0, end=0.05, decay_steps=int(TOTAL_STEPS * 0.6))

print(f"бюджет: {TOTAL_STEPS} шагов сбора, оценка каждые {TRAIN_CFG.eval_every_steps}")

бюджет: 2000 шагов сбора, оценка каждые 400


## 3. Эталон: Value Iteration на модели MDP

Динамическое программирование требует **знания модели** — и в этом его отличие
от Q-learning, которому модель не нужна. Модель здесь написана на Python
по тем же правилам, что и `GridWorldEnvironment.cs`, но независимо: если
реальная среда разойдётся с моделью хоть на одном переходе, значит одна из двух
реализаций неверна. Сверка выполняется в ячейке 5.

In [3]:
from labrl.algos.tabular.value_iteration import ValueIteration, ValueIterationConfig, optimality_rate
from labrl.envs.gridworld_mdp import GridWorldMDP

MDP = GridWorldMDP()
optimal = ValueIteration(ValueIterationConfig(gamma=ALGO_CFG.gamma)).solve(MDP)

print(f"сошлось за {optimal.iterations} итераций, последняя дельта {optimal.deltas[-1]:.2e}\n")
print("Оптимальная политика (# стена, G цель, X ловушка):")
print(MDP.render_policy(optimal.policy))
print("\nV*(s):")
print(MDP.render_values(optimal.values))

# Недисконтированная сумма наград оптимальной траектории — то самое число,
# с которым потом сравнивается Eval/Mean Reward.
s, total, steps = MDP.index(MDP.start), 0.0, 0
while steps < MDP.max_steps:
    s, r, term = MDP.step(s, int(optimal.policy[s]))
    total += r; steps += 1
    if term: break
print(f"\nОптимум: {steps} шагов, суммарная награда {total:.2f}")

сошлось за 10 итераций, последняя дельта 0.00e+00

Оптимальная политика (# стена, G цель, X ловушка):
 E  E  S  #  G 
 N  X  S  #  N 
 S  #  E  E  N 
 E  E  N  X  N 
 N  N  N  E  N 

V*(s):
  0.623   0.670   0.717   ####    0.000
  0.577   0.000   0.765   ####    0.960
  0.623   ####    0.813   0.861   0.910
  0.670   0.717   0.765   0.000   0.861
  0.623   0.670   0.717   0.765   0.813

Оптимум: 8 шагов, суммарная награда 0.68


## 4. Подключение среды и инспекция пространств

In [4]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E01_GridWorld"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True)
vec = VecUnityEnv(handle)
obs = vec.reset()

print(handle.describe())
print(f"\nарен (параллельных сред): {vec.num_envs}")

# Проверка соответствия ENV_SPEC.md: расхождение обязано падать здесь,
# а не проявляться странными результатами через час обучения.
assert vec.single_obs_dim == MDP.num_states, f"obs_dim {vec.single_obs_dim} != {MDP.num_states}"
assert tuple(vec.action_spec.discrete_branches) == (MDP.num_actions,)
print("размерности совпадают с ENV_SPEC.md")

behavior: E01_GridWorld?team=0
наблюдения (1 сенсоров):
  obs_0: shape=(25,)  имя сенсора: 'VectorSensor_size25'
действия: непрерывных 0, дискретные ветки (4,)

арен (параллельных сред): 4
размерности совпадают с ENV_SPEC.md


## 5. Сверка модели MDP с реальной средой

Прогоняем случайную политику и сравниваем каждый переход Unity с предсказанием
модели. Пустой список расхождений означает, что Python-реплика точна — и что
эталон из ячейки 3 применим к этой среде.

In [5]:
from labrl.envs.gridworld_mdp import check_against_env
from labrl.train.tabular import states_from_obs

rng = np.random.default_rng(SEED)
observed = []

for _ in range(200):
    state = states_from_obs(obs)
    action = rng.integers(0, MDP.num_actions, size=vec.num_envs)
    result = vec.step(action[:, None].astype(np.int32))

    done = result.terminated | result.truncated
    next_obs = np.where(done[:, None], result.final_obs[0], result.obs[0])
    next_state = np.argmax(next_obs, axis=1)

    for slot in np.flatnonzero(done | result.active):
        if result.truncated[slot]:
            continue  # обрыв по времени модель не описывает: это свойство эпизода
        observed.append((int(state[slot]), int(action[slot]), int(next_state[slot]),
                         float(result.reward[slot]), bool(result.terminated[slot])))
    obs = result.obs

problems = check_against_env(MDP, observed)
print(f"сверено переходов: {len(observed)}, расхождений: {len(problems)}")
for p in problems[:10]:
    print(" ", p)
assert not problems, "модель MDP разошлась с Unity — эталон недействителен"

сверено переходов: 800, расхождений: 0


## 6. Теория и математика метода

Q-learning хранит таблицу `Q(s, a)` и после каждого перехода сдвигает её
к цели Беллмана:

$$
y_t = r_t + \gamma\,(1 - \mathrm{terminated}_t)\,\max_{a'} Q(s_{t+1}, a')
$$

$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha\,[\,y_t - Q(s_t, a_t)\,]
$$

`max` по действиям делает метод **off-policy**: цель считается по жадной
политике, а опыт собирается ε-жадной. Замена `max` на `Q(s', a_{t+1})` дала бы
SARSA.

Множитель $(1-\mathrm{terminated})$ обнуляет будущее **только** при истинном
завершении. Обрыв по `MaxStep` (`truncated`) — не терминальное состояние: там
будущее есть. Ошибка здесь не роняет обучение, она тихо учит агента считать
нехватку времени катастрофой.

Ниже — **исходный код обновления**, тот самый, что выполняется при обучении.

In [6]:
import inspect

from labrl.algos.tabular.q_learning import QLearning

print(inspect.getsource(QLearning.update))

    def update(self, batch: Transition) -> dict[str, float]:
        """Единственное место, где меняется таблица Q (требование 8.7).

        Обработка повторов внутри батча. Если две арены дали переход из одной
        и той же пары (s, a), наивное векторное присваивание применит только
        последнее обновление — остальные потеряются. Поэтому приращения
        суммируются через ``np.add.at``: это эквивалентно последовательному
        применению всех переходов батча с одинаковым «старым» Q, что и есть
        стандартная семантика батчевого табличного обновления.

        Returns:
            Метрики шага обучения для TensorBoard.
        """
        state = np.asarray(batch.state, dtype=np.int64)
        action = np.asarray(batch.action, dtype=np.int64)
        reward = np.asarray(batch.reward, dtype=np.float64)
        next_state = np.asarray(batch.next_state, dtype=np.int64)
        terminated = np.asarray(batch.terminated, dtype=bool)

        if state.size == 0:
            

## 7. Цикл обучения

Цикл живёт в `labrl.train.tabular`, а не копируется сюда: иначе учебная
и рабочая версии со временем разойдутся (то же соображение, что и в 9.4
инструкции про алгоритм). Его исходник печатается ниже целиком — читать его
стоит до запуска.

In [7]:
from labrl.train.tabular import train_q_learning

print(inspect.getsource(train_q_learning))

def train_q_learning(
    vec: VecUnityEnv,
    algo: QLearning,
    epsilon_schedule: Schedule,
    logger: TBLogger,
    cfg: TabularTrainConfig | None = None,
    seed: int = 0,
    on_eval: Callable[[int, float, float], None] | None = None,
    lr_schedule: Schedule | None = None,
) -> TabularTrainResult:
    """Обучает табличный Q-learning в среде Unity.

    Args:
        vec: векторизованная среда (K арен = K параллельных сред).
        algo: алгоритм; его таблица меняется на месте.
        epsilon_schedule: расписание ε от номера шага сбора.
        logger: логгер TensorBoard.
        cfg: параметры цикла.
        seed: сид генератора выбора случайных действий.
        lr_schedule: расписание α. ``None`` — постоянное значение из конфига
            метода. Убывающее α требуется условием сходимости Роббинса–Монро
            (см. :meth:`labrl.algos.tabular.q_learning.QLearning.set_learning_rate`).
        on_eval: колбэк ``(step, mean_reward, success_rate)`` после каждой оценки 

In [8]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger

algo = QLearning(MDP.num_states, MDP.num_actions, ALGO_CFG)
run = create_run_dir(ENV_ID, "qlearning", SEED)
run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")

result = train_q_learning(
    vec=vec, algo=algo, epsilon_schedule=EPSILON, logger=logger,
    cfg=TRAIN_CFG, seed=SEED,
    on_eval=lambda step, r, s: print(f"  шаг {step:>6}: награда {r:+.4f}, успех {s:.0%}"),
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={"seed": SEED, "missing_required_tags": list(missing)})
logger.close()

print(f"\nэпизодов: {len(result.episode_returns)}, время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")

каталог прогона: C:\unity-ml-agents-lab\results\E01_GridWorld\qlearning\20260815-200436_seed0



  шаг    400: награда +0.6800, успех 100%


  шаг    800: награда +0.6800, успех 100%


  шаг   1200: награда +0.6800, успех 100%


  шаг   1600: награда +0.6800, успех 100%


  шаг   2000: награда +0.6800, успех 100%

эпизодов: 745, время: 9.5 с
обязательные теги схемы 11.2 без данных: нет


## 8. Оценка: сравнение с доказанным оптимумом

In [9]:
learned = algo.greedy_policy()
free = np.array([not MDP.is_terminal(s) and not MDP.is_wall(MDP.cell(s))
                 for s in range(MDP.num_states)])
rate = optimality_rate(learned, optimal.q, free)

print("Выученная политика:")
print(MDP.render_policy(learned))
print(f"\nдоля состояний с оптимальным действием: {rate:.1%}")
print(f"итоговая оценка: награда {result.last_eval_reward:+.4f}, успех {result.last_eval_success:.0%}")
print(f"оптимум среды:   награда {total:+.4f}")

# Награда обязана достигать оптимума в любом режиме: старт фиксирован,
# и траектория из него выучивается быстро.
assert result.last_eval_reward >= total - 1e-3, (
    f"награда {result.last_eval_reward:.4f} ниже оптимума {total:.4f}")

# А оптимальность ВО ВСЕХ состояниях требовать можно только от полного прогона.
# При QUICK_RUN бюджета не хватает, чтобы разведка добралась до клеток вдали
# от единственной стартовой траектории, и доля выходит около 80%. Это не
# деградация метода, а честное следствие сокращённого бюджета (требование 9.3).
if QUICK_RUN:
    print("\nQUICK_RUN: полная оптимальность не проверяется — не хватает бюджета на разведку")
else:
    assert rate == 1.0, "есть состояния, где выучено неоптимальное действие"
    print("\nполитика оптимальна во всех достижимых состояниях")

Выученная политика:
 N  E  S  #  G 
 S  X  S  #  N 
 S  #  E  E  N 
 S  S  S  X  N 
 E  E  E  E  N 

доля состояний с оптимальным действием: 78.9%
итоговая оценка: награда +0.6800, успех 100%
оптимум среды:   награда +0.6800

QUICK_RUN: полная оптимальность не проверяется — не хватает бюджета на разведку


## 9. Экспорт в ONNX и верификация

Таблица `Q` превращается в линейный слой без смещения: при one-hot наблюдении
`onehot(s) @ Qᵀ == Q[s]`, то есть слой возвращает ровно строку таблицы.
В Unity уезжает **та же** политика, а не её приближение.

Верификация блокирующая: любой провал означает, что пример не принят.

In [10]:
from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

spec = ActionSpecLite(discrete_branches=tuple(vec.action_spec.discrete_branches))
policy = algo.policy_module()
onnx_path = export_policy_to_onnx(policy, spec, vec.obs_shapes, run.onnx / "policy.onnx")

# Наблюдения из РЕАЛЬНОГО распределения среды: требование 10.5 говорит именно
# о нём. На случайном шуме расхождение нормализации может не проявиться.
sample = np.eye(MDP.num_states, dtype=np.float32)[
    np.random.default_rng(0).integers(0, MDP.num_states, size=64)]

verification = verify_onnx_model(onnx_path, policy, spec, vec.obs_shapes, sample_obs=[sample])
print(verification.report())
verification.raise_if_failed()

model_dest = REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID / "Models" / f"{ENV_ID}_qlearning.onnx"
model_dest.parent.mkdir(parents=True, exist_ok=True)
model_dest.write_bytes(onnx_path.read_bytes())
print(f"\nмодель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")

vec.close()

OK    структура (onnx.checker)
OK    opset — ожидался 9, в модели 9
OK    имена входов — лишние=[], отсутствуют=[]
OK    имена выходов — лишние=[], отсутствуют=[]
OK    тип выхода discrete_actions — ожидался целочисленный тензор, получен elem_type=7 (int64)
OK    тип выхода deterministic_discrete_actions — ожидался целочисленный тензор, получен elem_type=7 (int64)
OK    прогон onnxruntime, батч 1
OK    прогон onnxruntime, батч 64
OK    константа version_number — ожидалось [3.0], получено [3.0]
OK    константа memory_size — ожидалось [0.0], получено [0.0]
OK    константа discrete_action_output_shape — ожидалось [4.0], получено [4.0]
OK    числовой паритет: deterministic_discrete_actions — max|ONNX-PyTorch| = 0.000e+00, допуск 1e-04
OK    диапазон дискретной ветки 0 — значения [0, 2], допустимо целые [0, 4)
ИТОГ: ПРОЙДЕНО (13/13)

модель скопирована в проект Unity: unity\MLAgentsLab\Assets\Envs\E01_GridWorld\Models\E01_GridWorld_qlearning.onnx


## 10. Возврат в Unity

Модель уже лежит в `Assets/Envs/E01_GridWorld/Models/`. Дальше есть два пути.

### Автоматически (то, чем проверяется приёмка)

```powershell
python scripts\check_inference.py --config configs\E01_GridWorld__qlearning.yaml --python-reward 0.68
```

Скрипт соберёт билд с назначенной моделью и `Behavior Type = Inference Only`,
прогонит 20 эпизодов и сравнит награду с Python-оценкой. Критерий приёмки —
отношение ≥ 0.8 (требование 10.6). Фактически достигнуто **1.000**.

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.8f1.
2. Открыть сцену `Assets/Envs/E01_GridWorld/Scenes/E01_GridWorld.unity`.
3. Выделить модель в Project и выполнить **Tools → RL → Check ONNX Contract** —
   это подтверждает, что Inference Engine видит те же имена тензоров.
4. У агента любой арены в `Behavior Parameters`: `Model` = импортированная
   модель, `Behavior Type` = `Inference Only`.
5. Нажать Play: агент должен доходить до зелёной клетки за 8 шагов, обходя
   стены и красные ловушки.

### Просмотр метрик

```powershell
.\scripts\tb.ps1 -EnvId E01_GridWorld
```